# Chapter 10: Test Data Design

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Test Data Factory** that generates realistic, domain-specific test data sets based on requirements, schemas, and business rules. Includes data for positive, negative, and edge case scenarios.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Understanding of test data management


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Schema-Based Test Data Generation

Given a data schema or entity definition, generate realistic test data covering all scenario types.


In [ ]:
# Define the data schema
schema = {
    'entity': 'Customer Order',
    'fields': [
        {'name': 'order_id', 'type': 'string', 'format': 'ORD-YYYYMMDD-XXXX', 'required': True},
        {'name': 'customer_email', 'type': 'string', 'format': 'email', 'required': True},
        {'name': 'order_date', 'type': 'datetime', 'constraint': 'cannot be future date', 'required': True},
        {'name': 'items', 'type': 'array', 'min_items': 1, 'max_items': 50, 'required': True},
        {'name': 'total_amount', 'type': 'decimal', 'min': 0.01, 'max': 999999.99, 'required': True},
        {'name': 'currency', 'type': 'enum', 'values': ['USD', 'EUR', 'GBP', 'JPY'], 'required': True},
        {'name': 'shipping_address', 'type': 'object', 'required': True,
         'fields': ['street', 'city', 'state', 'zip', 'country']},
        {'name': 'discount_code', 'type': 'string', 'format': 'PROMO-XXXX', 'required': False},
        {'name': 'priority', 'type': 'enum', 'values': ['standard', 'express', 'overnight'], 'required': True}
    ],
    'business_rules': [
        'Overnight shipping only available for domestic US orders',
        'Discount codes give 10-30% off, cannot exceed total_amount',
        'Orders over $500 require manager approval',
        'JPY currency amounts must be whole numbers (no decimals)',
        'Maximum 3 orders per customer per day'
    ]
}

# Generate test data
data_prompt = f"""Generate test data for this schema. Create records for each scenario type.

Schema: {json.dumps(schema, indent=2)}

Generate these test data sets:
1. happy_path: 3 valid orders that pass all business rules
2. boundary: 5 records testing min/max limits of each field
3. negative: 5 records that should be rejected (one invalid field each, with a note about what's wrong)
4. edge_cases: 3 records with unusual but potentially valid scenarios

Return a JSON object with keys: happy_path, boundary, negative, edge_cases.
Each value is an array of order records.
For negative records, add a field 'expected_error' describing the expected validation failure.

Use realistic data (not 'test123' style). Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': data_prompt}],
    temperature=0.4
)
test_data = json.loads(response.choices[0].message.content)

for category, records in test_data.items():
    print(f"\n{category.upper()}: {len(records)} records")
    for r in records[:2]:
        print(f"  Order: {r.get('order_id', 'N/A')} | Amount: {r.get('total_amount', 'N/A')} {r.get('currency', '')}")
        if 'expected_error' in r:
            print(f"    Expected Error: {r['expected_error']}")

## 2. Business-Rule-Aware Data Generation

Generate data that specifically tests complex business rules and their interactions.


In [ ]:
# Generate a test data matrix for business rule coverage
matrix_prompt = f"""Create a test data matrix that ensures every business rule is tested.

Business Rules: {json.dumps(schema['business_rules'])}
Schema: {json.dumps(schema['fields'], indent=2)}

For each business rule:
1. Create a test record that satisfies the rule (positive test)
2. Create a test record that violates the rule (negative test)
3. Create a test record at the boundary of the rule

Return a JSON object with:
- matrix: array of {{rule, positive_record, negative_record, boundary_record}}
- combination_tests: 3 records that test multiple rules simultaneously

Use realistic, domain-appropriate data values. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': matrix_prompt}],
    temperature=0.3
)
matrix = json.loads(response.choices[0].message.content)

print('Business Rule Test Matrix:')
print('=' * 60)
for entry in matrix['matrix']:
    print(f"\nRule: {entry['rule']}")
    print(f"  Positive: order_id={entry['positive_record'].get('order_id', 'N/A')}")
    print(f"  Negative: order_id={entry['negative_record'].get('order_id', 'N/A')}")
    print(f"  Boundary: order_id={entry['boundary_record'].get('order_id', 'N/A')}")

In [ ]:
# Generate SQL insert statements for the test data
sql_prompt = f"""Convert these test data records into SQL INSERT statements for a PostgreSQL database.

Table name: customer_orders
Test Data (happy path only): {json.dumps(test_data['happy_path'])}

Also generate:
1. CREATE TABLE statement with appropriate column types and constraints
2. INSERT statements for the test data
3. Cleanup DELETE statements

Return the SQL as plain text."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': sql_prompt}],
    temperature=0.2
)
print('Generated SQL for test data setup:')
print(response.choices[0].message.content)

## Exercises
1. Generate test data for an API with nested objects (e.g., an order with line items, each with product details).
2. Create a data masking function that takes production data samples and anonymizes them for testing.
3. Build a test data generator that respects referential integrity across multiple tables.
4. Generate test data in multiple formats: JSON, CSV, SQL, and XML.
